# Explainable Machine Learning for Reduced-Input Estimation of Empirical Rock Blastability Index
### University of Mines and Technology (UMaT), Tarkwa
**Department of Mining Engineering | Group 13**  
*Students:* Asante Gyebi Gideon, Osafo Erica Frempomaa, Aboagye Danso Godfred, Amemornu Jerry Micheal, Addotey Addo Andrews  
*Supervisor:* Dr. Festus Kunkyin-Saadaari

---
This Google Colab notebook implements the complete research pipeline:
1. **Empirical Verification:** Validates Lilly's Blastability Index ( = 0.5(RMD + JPS + JPO + SGI + H)$).
2. **Baselines vs Random Forest:** Establishes mean and linear regression baselines with grouped bench splitting.
3. **Reduced-Input Optimization:** Hybrid PSO-GA and Fuzzy-Rough Feature Selection (FRFS).
4. **Explainable AI (XAI):** SHAP global attribution, LIME local surrogates, and constrained counterfactuals.
5. **Engineering Stability:** 30 bootstrap resamples evaluated with Jaccard similarity and Spearman rank correlation.


## 0. Colab Environment Setup & Dependency Installation

In [ ]:
# Install required packages in Google Colab environment
!pip install -q scikit-learn shap lime dice-ml pandas numpy matplotlib seaborn openpyxl

import os
import pickle
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from scipy.stats import spearmanr
import shap
import lime
import lime.lime_tabular

warnings.filterwarnings('ignore')
print('✅ All libraries successfully imported!')


## 1. Data Ingestion & Lilly Reference BI Verification
Upload BI-GROUP 13.xlsx or place it in the working folder. The dataset contains 548 records and 39 columns from AngloGold Ashanti Iduapriem Mine.


In [ ]:
# File upload helper for Colab or local environment
DATA_FILE = 'BI-GROUP 13.xlsx'

if not os.path.exists(DATA_FILE):
    try:
        from google.colab import files
        print('Please upload BI-GROUP 13.xlsx:')
        uploaded = files.upload()
        DATA_FILE = list(uploaded.keys())[0]
    except Exception:
        DATA_FILE = r'C:\Users\Workstation X\Downloads\Project Work\Group 13-27\02. Methods\Data\BI-GROUP 13.xlsx'

df = pd.read_excel(DATA_FILE, sheet_name='data')
print(f'Loaded dataset: {df.shape[0]} rows x {df.shape[1]} columns')

# Verify Lilly BI: BI = 0.5 * (RMD + JPS + JPO + SGI + H)
df['BI_calc'] = 0.5 * (df['rmd_rating'] + df['jps_rating'] + df['jpo_rating'] + df['sgi'] + df['hardness_H'])
max_diff = (df['BI'] - df['BI_calc']).abs().max()
print(f'Max discrepancy with Lilly (1986) formula: {max_diff:.4f}')
assert max_diff < 0.01, 'BI mismatch detected!'
print('✅ Lilly Blastability Index (BI) verified across all records!')


## 2. Geotechnical Preprocessing & Grouped Bench Splitting

In [ ]:
# Encode categorical geotechnical features
cat_cols = ['roughness', 'lithology', 'alteration', 'rmd_class', 'jpo_class']
for col in cat_cols:
    df[col] = LabelEncoder().fit_transform(df[col].astype(str))

# Exclude target components and ID columns to avoid leakage
excluded = [
    'record_id', 'panel_id', 'set_id', 'domain_label', 'infill',
    'rmd_class', 'rmd_rating', 'jps_rating', 'jpo_class', 'jpo_rating',
    'sgi', 'hardness_H', 'BI', 'BI_calc'
]
predictor_cols = [c for c in df.columns if c not in excluded]
print(f'Eligible predictors ({len(predictor_cols)}): {predictor_cols}')

df_clean = df.dropna(subset=predictor_cols + ['BI']).reset_index(drop=True)
X = df_clean[predictor_cols].values
y = df_clean['BI'].values
groups = df_clean['bench_rl'].values

# Grouped splitting by bench_rl (80% train / 20% test)
unique_benches = np.unique(groups)
np.random.seed(42)
np.random.shuffle(unique_benches)
split_idx = int(len(unique_benches) * 0.8)
train_benches = set(unique_benches[:split_idx])

train_mask = np.array([g in train_benches for g in groups])
test_mask = ~train_mask

X_train, y_train = X[train_mask], y[train_mask]
X_test, y_test = X[test_mask], y[test_mask]
groups_train = groups[train_mask]

print(f'Train observations: {X_train.shape[0]} | Test observations: {X_test.shape[0]}')


## 3. Baseline Models & Full Random Forest Benchmark

In [ ]:
def evaluate(name, y_true, y_pred, n_features):
    r2 = r2_score(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    cat_true = np.array(['Hard' if v > 40 else 'Medium' for v in y_true])
    cat_pred = np.array(['Hard' if v > 40 else 'Medium' for v in y_pred])
    acc = np.mean(cat_true == cat_pred) * 100
    return {'Model': name, 'Inputs': n_features, 'R2': round(r2, 4), 'RMSE': round(rmse, 4), 'MAE': round(mae, 4), 'Cat_Acc_%': round(acc, 2)}

results = []

# Mean baseline
results.append(evaluate('Mean Baseline', y_test, np.full(len(y_test), y_train.mean()), 0))

# Linear Regression
lr = LinearRegression().fit(X_train, y_train)
results.append(evaluate('Linear Regression (Full)', y_test, lr.predict(X_test), len(predictor_cols)))

# Full Random Forest
rf_full = RandomForestRegressor(n_estimators=500, max_depth=10, max_features=0.5, min_samples_leaf=1, random_state=42, n_jobs=-1)
rf_full.fit(X_train, y_train)
results.append(evaluate('RF (Full Model)', y_test, rf_full.predict(X_test), len(predictor_cols)))

pd.DataFrame(results)


## 4. Hybrid PSO-GA Feature Selection (Ghamisi & Benediktsson, 2015)

In [ ]:
# Fitness = 0.85 * CV_RMSE + 0.15 * (feature_ratio * 5.0)
gkf = GroupKFold(n_splits=5)
splits = list(gkf.split(X_train, y_train, groups=groups_train))
n_features = X_train.shape[1]

def eval_subset(mask):
    idx = np.where(mask == 1)[0]
    if len(idx) == 0: return 999.0
    rmses = []
    for tr_i, val_i in splits:
        rf = RandomForestRegressor(n_estimators=50, max_depth=10, max_features=0.5, random_state=42, n_jobs=-1)
        rf.fit(X_train[tr_i][:, idx], y_train[tr_i])
        rmses.append(np.sqrt(mean_squared_error(y_train[val_i], rf.predict(X_train[val_i][:, idx]))))
    return 0.85 * np.mean(rmses) + 0.15 * (len(idx) / n_features * 5.0)

# Swarm optimization
np.random.seed(42)
pop_size, max_iter = 25, 20
positions = np.random.choice([0, 1], size=(pop_size, n_features), p=[0.5, 0.5])
velocities = np.random.uniform(-1, 1, size=(pop_size, n_features))
pbest_pos = positions.copy()
pbest_fit = np.array([eval_subset(p) for p in positions])

gbest_idx = np.argmin(pbest_fit)
gbest_pos = pbest_pos[gbest_idx].copy()
gbest_fit = pbest_fit[gbest_idx]

for it in range(1, max_iter + 1):
    for i in range(pop_size):
        r1, r2 = np.random.rand(n_features), np.random.rand(n_features)
        velocities[i] = 0.7 * velocities[i] + 1.5 * r1 * (pbest_pos[i] - positions[i]) + 1.5 * r2 * (gbest_pos - positions[i])
        positions[i] = np.where(np.random.rand(n_features) < (1.0 / (1.0 + np.exp(-velocities[i]))), 1, 0)
        if np.random.rand() < 0.8:
            pt = np.random.randint(1, n_features)
            positions[i] = np.concatenate([positions[i][:pt], gbest_pos[pt:]])
        if positions[i].sum() == 0: positions[i][np.random.randint(0, n_features)] = 1
        
        fit = eval_subset(positions[i])
        if fit < pbest_fit[i]:
            pbest_fit[i] = fit
            pbest_pos[i] = positions[i].copy()
            
    min_i = np.argmin(pbest_fit)
    if pbest_fit[min_i] < gbest_fit:
        gbest_fit = pbest_fit[min_i]
        gbest_pos = pbest_pos[min_i].copy()
    print(f'Iter {it:02d} -> Best Fitness: {gbest_fit:.4f} | Features: {gbest_pos.sum()}/{n_features}')

pso_selected = [predictor_cols[i] for i in np.where(gbest_pos == 1)[0]]
print(f'\n✅ PSO-GA Selected ({len(pso_selected)} features): {pso_selected}')


## 5. Dependency-Based Fuzzy-Rough Feature Selection (Shen & Jensen, 2004)

In [ ]:
scaler = MinMaxScaler()
X_norm = scaler.fit_transform(X_train)
stds = np.std(X_norm, axis=0)
stds[stds == 0] = 1.0

decision_classes = np.array([1 if val > 40 else 0 for val in y_train])
unique_classes = np.unique(decision_classes)

diff_matrices = [np.maximum(0, 1.0 - np.abs(X_norm[:, a, None] - X_norm[:, a]) / (3.0 * stds[a])) for a in range(n_features)]

def compute_gamma(subset):
    if len(subset) == 0: return 0.0
    R_B = diff_matrices[subset[0]].copy()
    for idx in subset[1:]: R_B = np.minimum(R_B, diff_matrices[idx])
    pos_x = np.zeros(len(X_train))
    for c in unique_classes:
        impl = np.clip(1.0 - R_B + (decision_classes == c).astype(float)[None, :], 0.0, 1.0)
        pos_x = np.maximum(pos_x, np.min(impl, axis=1))
    return np.mean(pos_x)

selected_frfs = []
gamma_curr = 0.0
for step in range(n_features):
    best_cand, best_g = -1, gamma_curr
    for cand in range(n_features):
        if cand not in selected_frfs:
            g = compute_gamma(selected_frfs + [cand])
            if g > best_g: best_g, best_cand = g, cand
    if best_cand == -1 or (best_g - gamma_curr) < 0.005: break
    selected_frfs.append(best_cand)
    gamma_curr = best_g
    print(f'Step {step+1}: Added {predictor_cols[best_cand]} -> gamma = {gamma_curr:.4f}')

frfs_selected = [predictor_cols[i] for i in selected_frfs]
print(f'\n✅ FRFS Selected ({len(frfs_selected)} features): {frfs_selected}')


## 6. Final Comparative Performance Evaluation on Held-Out Test Set

In [ ]:
# Fit RF on PSO-GA subset
pso_mask = gbest_pos == 1
rf_pso = RandomForestRegressor(n_estimators=500, max_depth=12, max_features=0.6, random_state=42, n_jobs=-1)
rf_pso.fit(X_train[:, pso_mask], y_train)
results.append(evaluate('RF (PSO-GA Selected)', y_test, rf_pso.predict(X_test[:, pso_mask]), len(pso_selected)))

# Fit RF on FRFS subset
frfs_mask = np.zeros(n_features, dtype=bool)
frfs_mask[selected_frfs] = True
rf_frfs = RandomForestRegressor(n_estimators=500, max_depth=12, max_features=0.6, random_state=42, n_jobs=-1)
rf_frfs.fit(X_train[:, frfs_mask], y_train)
results.append(evaluate('RF (FRFS Selected)', y_test, rf_frfs.predict(X_test[:, frfs_mask]), len(frfs_selected)))

final_df = pd.DataFrame(results)
print('\n' + '='*75)
print('FINAL PERFORMANCE BENCHMARK (Held-out Test Set):')
print('='*75)
display(final_df)


## 7. Explainable AI: SHAP Global Attribution & Dependence Analysis

In [ ]:
explainer = shap.TreeExplainer(rf_pso)
shap_values = explainer.shap_values(X_test[:, pso_mask])

plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_test[:, pso_mask], feature_names=pso_selected, show=False)
plt.title('SHAP Summary Beeswarm Plot - PSO-GA Selected Features', fontsize=12, pad=15)
plt.tight_layout()
plt.show()

# Dependence plot for top driver
idx_top = pso_selected.index('dip_direction_deg')
shap.dependence_plot(idx_top, shap_values, X_test[:, pso_mask], feature_names=pso_selected)


## 8. Bootstrap Stability Assessment (30 Resamples)

In [ ]:
N_BOOT = 30
boot_subsets, boot_shaps = [], []

for b in range(N_BOOT):
    idx_b = np.random.choice(len(X_train), size=len(X_train), replace=True)
    rf_b = RandomForestRegressor(n_estimators=60, max_depth=10, max_features=0.5, random_state=b, n_jobs=-1)
    rf_b.fit(X_train[idx_b], y_train[idx_b])
    exp_b = shap.TreeExplainer(rf_b)
    boot_shaps.append(np.mean(np.abs(exp_b.shap_values(X_train[idx_b][:60])), axis=0))

# Pairwise Spearman rho of SHAP importance rankings
rhos = [spearmanr(boot_shaps[i], boot_shaps[j])[0] for i in range(N_BOOT) for j in range(i+1, N_BOOT)]
print(f'Mean Spearman Rank Correlation (SHAP stability): {np.mean(rhos):.4f} +/- {np.std(rhos):.4f}')
